# Fashion-MNIST Image Classification with a Neural Network

A beginner-friendly deep-learning project that trains a fully connected neural network to classify grayscale clothing images into 10 categories using TensorFlow/Keras.

## Project objectives
- Load and explore the Fashion-MNIST dataset.
- Normalize image pixel values.
- Build and train a neural network with a `Flatten` layer and dense layers.
- Evaluate the model on the test set.
- Inspect predictions and optionally classify a custom 28×28 grayscale image.

**Dataset:** Fashion-MNIST, containing 28×28 grayscale images across 10 clothing categories.

## 1. Imports and reproducibility

The code uses TensorFlow/Keras for model training, Matplotlib for visualization, and NumPy for numerical operations. OpenCV is only needed for the optional custom-image example.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Reproducibility: results can still vary slightly across hardware and library versions.
tf.keras.utils.set_random_seed(42)

print("TensorFlow version:", tf.__version__)

## 2. Load the dataset

Keras downloads Fashion-MNIST on the first run if it is not already cached. The dataset provides 60,000 training images and 10,000 test images.

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Training images:", train_images.shape)
print("Test images:", test_images.shape)
print("Pixel range before normalization:", train_images.min(), "to", train_images.max())

## 3. Explore and preprocess the images

Each image is 28×28 pixels. Dividing pixel values by 255 scales them from `[0, 255]` to `[0, 1]`, which is a common preprocessing step for neural networks.

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(train_images[0], cmap="gray")
plt.title(f"Label: {class_names[train_labels[0]]}")
plt.axis("off")
plt.show()

train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0

print("Pixel range after normalization:", train_images.min(), "to", train_images.max())

## 4. Build the model

`Flatten` converts each 28×28 image into a vector of 784 values. The hidden dense layer learns features, and the output layer returns 10 class scores (logits), one for each category.

In [ ]:
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation="relu"),
    Dense(10)
])

model.summary()

## 5. Compile and train

`SparseCategoricalCrossentropy(from_logits=True)` is appropriate because the labels are integer class IDs and the model outputs logits. Adam is used as the optimizer, and accuracy is tracked during training.

In [ ]:
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history = model.fit(
    train_images,
    train_labels,
    epochs=10,
    validation_split=0.1,
    verbose=1
)

## 6. Review training progress

The following plots help compare loss and accuracy across epochs. Validation metrics are calculated on the validation split reserved by `model.fit`.

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history.history["loss"], label="Training loss")
plt.plot(epochs, history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss by epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history.history["accuracy"], label="Training accuracy")
plt.plot(epochs, history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy by epoch")
plt.legend()

plt.tight_layout()
plt.show()

## 7. Evaluate on the test set

The test set is held out from training and provides a more independent estimate of model performance.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_images, test_labels, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")

## 8. Inspect predictions

Because the model returns logits, apply softmax to convert them into class probabilities. The example below displays a test image, the predicted category, and the model's confidence.

In [ ]:
probability_model = Sequential([
    model,
    tf.keras.layers.Softmax()
])

sample_index = 4
image = test_images[sample_index]
true_label = int(test_labels[sample_index])

probabilities = probability_model.predict(image[np.newaxis, ...], verbose=0)[0]
predicted_label = int(np.argmax(probabilities))

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray")
plt.title(
    f"Predicted: {class_names[predicted_label]} ({probabilities[predicted_label]:.1%})\n"
    f"Actual: {class_names[true_label]}"
)
plt.axis("off")
plt.show()

print("Class probabilities:")
for name, probability in zip(class_names, probabilities):
    print(f"{name:12s}: {probability:.3f}")

## 9. Optional: classify a custom image

To try your own image, place it at `data/custom_image.png` (or update `image_path`). It should be a **28×28 grayscale image** with a visual style similar to Fashion-MNIST. This cell is optional and safely skips prediction if the file is absent.

**Important:** A photo or image with a different background, scale, or contrast may need preprocessing (including background inversion) before its prediction is meaningful.

In [ ]:
from pathlib import Path

image_path = Path("data/custom_image.png")

if image_path.exists():
    import cv2

    custom_image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if custom_image is None:
        raise ValueError(f"Could not read image: {image_path}")

    if custom_image.shape != (28, 28):
        raise ValueError(
            f"Expected a 28x28 image, but received {custom_image.shape}. "
            "Resize and preprocess it before prediction."
        )

    custom_image = custom_image.astype("float32") / 255.0
    custom_probabilities = probability_model.predict(
        custom_image[np.newaxis, ...], verbose=0
    )[0]
    custom_prediction = int(np.argmax(custom_probabilities))

    plt.figure(figsize=(4, 4))
    plt.imshow(custom_image, cmap="gray")
    plt.title(
        f"Prediction: {class_names[custom_prediction]} "
        f"({custom_probabilities[custom_prediction]:.1%})"
    )
    plt.axis("off")
    plt.show()
else:
    print(f"Optional image not found: {image_path}")
    print("Add a 28x28 grayscale image there to run custom-image inference.")

## Limitations and next steps

- This is a baseline fully connected neural network, not a convolutional neural network (CNN).
- The custom-image example expects an image already prepared to match the dataset's 28×28 grayscale format; arbitrary real-world images may not work well without additional preprocessing.
- Useful extensions include a confusion matrix, per-class precision/recall, saving and loading the trained model, and comparing the baseline with a CNN.

## Summary

This notebook demonstrates a complete introductory classification workflow: data loading, preprocessing, model definition, training, validation, test evaluation, and inference.